In [8]:
import pandas as pd

In [9]:
# Histórico global de jobs gerenciados pelo YARN
jobs = []

def enviar_job(nome, funcao):
    print(f"[YARN] Iniciando job: {nome}")
    resultado = funcao()
    jobs.append({
        'nome': nome,
        'status': 'finalizado',
        'resultado': resultado
    })
    return resultado

In [10]:
# Job 1: Média/Soma de vendas agrupadas por faixa etária do cliente
def job_vendas_por_faixa_etaria():
    df_v = pd.read_csv('../vendas.csv')
    df_c = pd.read_csv('../clientes.csv')

    df_merged = df_v.merge(df_c, on='id_cliente', how='inner')
    limites = [0, 18, 30, 45, 60, 100]
    rotulos = ['0-18', '19-30', '31-45', '46-60', '61+']

    df_merged["FaixaEtaria"] = pd.cut(df_merged["idade"], bins=limites, labels=rotulos)
    resultado = df_merged.groupby("FaixaEtaria", observed=False)['quantidade'].sum().to_dict()
    return resultado

In [13]:
# Job 2: Análise de valores em aberto no crediário por faixa de atraso
def job_inadimplencia_crediario():
    df_cred = pd.read_csv('../crediario.csv')
    limites = [-1, 0, 30, 60, 90, 999]
    rotulos = ['Em dia', '1-30 dias', '31-60 dias', '61-90 dias', '90+ dias']

    df_cred['FaixaAtraso'] = pd.cut(df_cred['dias_atraso'], bins=limites, labels=rotulos)
    resultado = df_cred.groupby('FaixaAtraso', observed=False)['valor_em_aberto'].sum().round(2).to_dict()
    return resultado

In [14]:
# Submissão dos jobs ao YARN
res_faixa = enviar_job("Total de Vendas por Faixa Etária", job_vendas_por_faixa_etaria)
res_cred = enviar_job("Inadimplência do Crediário por Atraso", job_inadimplencia_crediario)

[YARN] Iniciando job: Total de Vendas por Faixa Etária
[YARN] Iniciando job: Inadimplência do Crediário por Atraso


In [15]:
print("\n[YARN] Histórico de execução de Jobs:")
for job in jobs:
    print(f"- {job['nome']} | Status: {job['status']}")


[YARN] Histórico de execução de Jobs:
- Total de Vendas por Faixa Etária | Status: finalizado
- Total de Vendas por Faixa Etária | Status: finalizado
- Inadimplência do Crediário por Atraso | Status: finalizado
